# #30DayMapChallenge
## Day 24: Places and Their Names

In [3]:
import osmnx as ox

In [1]:
# highway for roads
tags = {"highway": True}

gdf = ox.features_from_place("Singapore", tags=tags)

In [5]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 292924 entries, ('node', 25455287) to ('way', 469394534)
Columns: 470 entries, highway to crossway
dtypes: geometry(1), object(469)
memory usage: 1.0+ GB


### Language Test

Results are not very accurate and generate a lot of noise.

In [42]:
import fasttext
import urllib.request

In [40]:
url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz"
urllib.request.urlretrieve(url, "lid.176.ftz")

('lid.176.ftz', <http.client.HTTPMessage at 0x16fa4513510>)

In [11]:
model = fasttext.load_model("lid.176.ftz")

In [44]:
def fasttext_lang(text):
    if not isinstance(text, str) or text.strip() == "":
        return None
    
    label, prob = model.predict(text)
    lang = label[0].replace("__label__", "")
    return lang

In [46]:
gdf["language"] = gdf["name"].apply(fasttext_lang)

In [50]:
gdf["language"].value_counts(dropna=False)

language
None    206261
en       68137
ms        4631
id        2609
it        2033
ceb       1743
hu        1296
es        1293
fr        1102
de         911
pt         517
ro         294
nl         275
da         265
fi         259
ca         233
no         176
eo         151
la         142
sv          89
pl          83
et          75
war         59
lt          36
eu          28
jv          27
cs          26
cy          20
zh          17
tr          17
vi          15
el          13
lv          13
mk          11
ja          10
hy           8
az           5
mr           5
sk           5
ru           4
oc           3
fa           3
ml           3
th           3
ar           3
sh           3
km           1
tl           1
sl           1
as           1
sr           1
is           1
hi           1
gl           1
ta           1
min          1
ckb          1
hr           1
Name: count, dtype: int64

In [52]:
# Show sample road names for the top languages
top = [
    'en', 'ms', 'id', 'it', 'ceb'
]

for lang in top:
    print(f"\n=== Language: {lang} ===")
    sample = gdf[gdf['language'] == lang]['name'].dropna().head(5)
    if len(sample) == 0:
        print("  (no examples found)")
    else:
        for s in sample:
            print(" •", s)


=== Language: en ===
 • Kallang
 • Lower Delta
 • West Coast
 • Boon Lay
 • Lower Delta

=== Language: ms ===
 • Bukit Merah
 • Before Telok Blangah Hill Park
 • Bukit Merah Interchange
 • Before Jalan Buroh
 • Before Telok Blangah Heights

=== Language: id ===
 • After Jalan Pesawat
 • After Kempas Road
 • Before Marinteknik
 • Bras Basah Complex
 • Palawan Beach

=== Language: it ===
 • Buona Vista
 • Clementi Flyover
 • Buona Vista Terminal
 • After Clementi Road
 • Before South Buona Vista Road

=== Language: ceb ===
 • bus stop geylang serai (datang)
 • Ang Siang Hill
 • Birdz of Play
 • Ang Siang Hill
 • Ang Siang Hill


### Categorisation by Street and Building Name Board - Handbook on Guidelines for Naming of Street

In [62]:
def classify_hierarchy(name):
    n = str(name).lower()
    if any(word in n for word in ["expressway", "highway", "parkway"]):
        return "Expressway"
    if any(word in n for word in ["boulevard", "avenue", "way"]):
        return "Major Arterial"
    if any(word in n for word in ["drive", "street", "road"]):
        return "Arterial/Primary Access"
    if any(word in n for word in ["walk", "lane", "link"]):
        return "Local Access"
    return None

def classify_configuration(name):
    n = str(name).lower()
    if any(word in n for word in ["oval", "circle", "circus", "circuit", "ring", "loop", "crescent"]):
        return "Loop/Oval"
    if any(word in n for word in ["bow", "square", "court", "close", "junction", "cross", "turn"]):
        return "Square/Court"
    return None

def classify_topography(name):
    n = str(name).lower()
    
    high_words = ["mount", "hill", "ridge", "peak", "crest", "heights", "rise"]
    low_words  = ["basin", "vale", "valley", "plain", "field", "park", "garden", "green", "grove"]
    
    if any(word in n for word in high_words):
        return "High"
    elif any(word in n for word in low_words):
        return "Low"
    else:
        return "None"

def is_water_proximity(name):
    n = str(name).lower()
    return any(word in n for word in ["cove", "bay", "marine", "marina", "quay", "river", 
                                      "bayshore", "bayfront", "sea", "ocean", "harbourfront", 
                                      "straits", "coast"])

def is_locational(name):
    n = str(name).lower()
    return any(word in n for word in ["central", "centre", "gate", "gateway", "north", "south", 
                                      "east", "west", "upper", "lower", "old", "new", "first", 
                                      "second", "third", "view", "vista", "point", "perimeter", 
                                      "boundary", "edge", "sector", "terrace", "concourse", 
                                      "esplanade", "promenade", "parade", "place", "estate", 
                                      "town", "village"])

def is_function(name):
    n = str(name).lower()
    return any(word in n for word in ["bridge", "causeway", "boardwalk", "flyover", "tunnel", 
                                      "underpass", "viaduct"])

In [64]:
# Apply to GeoDataFrame
gdf["hierarchy"] = gdf["name"].apply(classify_hierarchy)
gdf["configuration"] = gdf["name"].apply(classify_configuration)
gdf['topography'] = gdf['name'].apply(classify_topography)
gdf["water_proximity"] = gdf["name"].apply(is_water_proximity)
gdf["locational"] = gdf["name"].apply(is_locational)
gdf["function"] = gdf["name"].apply(is_function)

In [66]:
# Select key columns
key_columns = [
    "name",            # street name
    "hierarchy",       # street hierarchy
    "configuration",   # configuration/topology
    "topography",      # topography (bool)
    "water_proximity", # water proximity (bool)
    "locational",      # locational/contextual (bool)
    "function",        # function/infrastructure (bool)
    "geometry"         # keep geometry for mapping
]

# Create slim GeoDataFrame
gdf_slim = gdf[key_columns]

# Export to GeoPackage
gdf_slim.to_file("singapore_roads_classified.gpkg", layer="roads_classified", driver="GPKG")